In [3]:
import os
from matplotlib import pyplot as plt
import numpy as np
import sklearn
import sklearn.model_selection
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [4]:
# load class 1 data
data_path = 'data'

class1_data_path = os.path.join(data_path, 'class1')
class2_data_path = os.path.join(data_path, 'class2')

fix_size = 35_000


def load_file_data(file_data_path):
    
    if not os.path.isfile(file_data_path):
        return None
    
    data = np.array([])
    with open(file_data_path, 'r') as data_file:
        data = np.loadtxt(data_file, delimiter=' ')
        
    if data.shape[0] > fix_size:
        data = data[:fix_size, :]
    else:
        x = fix_size // data.shape[0]
        for i in range(x):
            data = np.concatenate((data, data), axis=0)

    data = data[:fix_size]
    return data


def load_data(folder_data_path):

    result_data = np.array([])
    for data_file_name in os.listdir(folder_data_path):
        
        data_file_path = os.path.join(folder_data_path, data_file_name)
        data = load_file_data(data_file_path)
        result_data = np.append(result_data, data)

    x = len(result_data) // fix_size
    result_data = result_data.reshape((x, fix_size))
    return result_data

class1_data = load_data(class1_data_path)
class1_label = np.ones((class1_data.shape[0]))

class2_data = load_data(class2_data_path)
class2_label = np.zeros((class2_data.shape[0]))

print(class1_data.shape)
print(class2_data.shape)
print(class1_label.shape)
print(class2_label.shape)


data = np.concatenate((class1_data, class2_data), axis=0)
print(data.shape)
label = np.concatenate((class1_label, class2_label), axis=0)
print(label.shape)

print(data[0])
print(label[0])


(38, 35000)
(73, 35000)
(38,)
(73,)
(111, 35000)
(111,)
[-10985. -11040. -11111. ... -10379. -10472. -10204.]
1.0


In [5]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(data, label, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

(88, 35000)
(23, 35000)


In [6]:

# Path: model.ipynb
from sklearn import svm
from sklearn.metrics import accuracy_score

clf = svm.SVC()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(accuracy_score(y_test, y_pred))


0.6521739130434783


In [8]:
# Use CNN to train model
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt


# Path: model.ipynb
size = X_train.shape[0] * X_train.shape[1]

X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print(X_train.shape)
print(X_test.shape)


model = models.Sequential(
    [
        layers.Conv1D(32, 3, activation="relu", input_shape=(size, 1)),
        layers.MaxPooling1D(2),
        layers.Conv1D(64, 3, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, 3, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.show()



(88, 35000, 1)
(23, 35000, 1)


2023-05-09 15:46:55.361296: W tensorflow/tsl/framework/bfc_allocator.cc:485] Allocator (mklcpu) ran out of memory trying to allocate 23.50GiB (rounded to 25231228928)requested by op StatelessRandomUniformV2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2023-05-09 15:46:55.361357: I tensorflow/tsl/framework/bfc_allocator.cc:1039] BFCAllocator dump for mklcpu
2023-05-09 15:46:55.361381: I tensorflow/tsl/framework/bfc_allocator.cc:1046] Bin (256): 	Total Chunks: 0, Chunks in use: 0. 0B allocated for chunks. 0B in use in bin. 0B client-requested in use in bin.
2023-05-09 15:46:55.361397: I tensorflow/tsl/framework/bfc_allocator.cc:1046] Bin (512): 	Total Chunks: 0, Chunks in use: 0. 0B allocated for chunks. 0B in use in bin. 0B client-requested in use in bin.
2023-05-09 15:46:55.361407: I tensorflow/tsl/framework/bfc_allocator.cc:10

ResourceExhaustedError: {{function_node __wrapped__StatelessRandomUniformV2_device_/job:localhost/replica:0/task:0/device:CPU:0}} OOM when allocating tensor with shape[49279744,128] and type float on /job:localhost/replica:0/task:0/device:CPU:0 by allocator mklcpu [Op:StatelessRandomUniformV2]